### Setup & configuration

In [ ]:
import os
import sys
import importlib
import json
import time
import mlflow
import pandas as pd
import torch
import importlib
from IPython.display import display, Markdown, JSON, HTML
# append the parent folder to the system path to allow imports from there
sys.path.append('..')

### INPUT VARIABLES

In [7]:
MODEL_NAME = "../../model/Qwen2.5-1.5B-Instruct"
EMB_MODEL_NAME = "../../model/bge-large-en-v1.5"

# define the absolute path of the LLM model
MODEL_PATH = os.path.abspath(MODEL_NAME)
EMB_MODEL_PATH = os.path.abspath(EMB_MODEL_NAME)

In [ ]:
worklog= """
INC-99102

System monitoring detected intermittent packet loss on core switch cluster in Toronto data node TD-14.

Initial alert triggered at 03:41 AM when ICMP ping failures exceeded threshold (packet loss > 40%) for IP range 10.22.14.0/24.

NOC engineer report:
Multiple devices intermittently responding. Device SW-CORE-19 shows unstable connectivity. SSH access failed twice.

At 03:58 AM, logs show repeated ICMP timeout for 10.22.14.23 and 10.22.14.31.

A technician was dispatched but initial remote diagnostics were inconclusive due to network flapping.

On-site technician report (Alex P.):
Device SW-CORE-19 completely unresponsive to ping and console access.
Physical inspection revealed NIC card failure suspected after overheating warning logs.

Replacement procedure initiated:
- removed faulty network interface card
- installed replacement NIC module
- rebooted device

By 05:10 AM, device restored and stable ping response confirmed.

Customer impact: intermittent service degradation across VLAN 220-245.

Next action: monitor stability for 24 hours and confirm no packet loss recurrence.

"""

# I. HF Models Prompting

### setup MLflow

In [8]:
import mlflow

mlflow.set_experiment("hf-based-prompting")


<Experiment: artifact_location='/media/zirox2025/DATA/running/GitHub/Knowledge-base-Extraction/gcp-evaluator/tutorials/mlruns/2', creation_time=1779034370168, experiment_id='2', last_update_time=1779034370168, lifecycle_stage='active', name='hf-based-prompting', tags={}, trace_location=None, workspace='default'>

### Instantiate the LLM model

In [ ]:
# import os
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # define the absolute path of the LLM model
# MODEL_PATH = os.path.abspath(MODEL_NAME)
# print(f"Loading model {MODEL_PATH}...")

# # instantiate the model and tokenizer with local_files_only=True to ensure it loads from the local path
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH,
#     torch_dtype="auto",
#     device_map="auto",
#     local_files_only=True,
# )

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_PATH,
#     local_files_only=True,
# )

# model.eval()

### Run the experiment

In [9]:
PROMPT = """
You are a high-fidelity industrial incident reconstruction system.

Extract a complete structured incident representation.

CRITICAL RULES:
- Preserve all telemetry
- Preserve all IPs
- Preserve all device identifiers
- Preserve all technician actions
- Never hallucinate
- Never merge unrelated events
- Each event must be atomic

WORKLOG:
{worklog}

Return structured extraction only.
"""

In [ ]:
import importlib

# load modules/prompts
import main
importlib.reload(main)
from main import load_llm, extract, verify, enrich

model_name = os.path.basename(MODEL_NAME)

with mlflow.start_run():

    start = time.time()

    llm = load_llm(MODEL_PATH)

    extraction = extract(llm, worklog)
    verification = verify(llm, worklog, extraction)
    final = enrich(llm, worklog, extraction, verification)

    latency = time.time() - start

    # =====================================================
    # MLflow logging
    # =====================================================

    mlflow.log_param("model_name", model_name)
    mlflow.log_metric("latency_sec", latency)
    mlflow.log_metric("fix_required", int(verification.fix_required))

    mlflow.log_text(worklog, "input.txt")
    mlflow.log_text(extraction.model_dump_json(indent=2), "extraction.json")
    mlflow.log_text(verification.model_dump_json(indent=2), "verification.json")
    mlflow.log_text(final.model_dump_json(indent=2), "final.json")

    print(final.model_dump_json(indent=2))

Loading weights: 100%|██████████| 338/338 [00:03<00:00, 107.36it/s]


# II. LLM Summary Evaluation

### setup MLflow

In [ ]:
import mlflow

mlflow.set_experiment("hf-based-evaluation")


### Instantiate the LLM model

In [ ]:
import json
import time
import os
import pandas as pd
import torch
import importlib
# load modules/prompts
import llm_evaluation

importlib.reload(llm_evaluation)
from llm_evaluation import HybridLLMEvaluator

MODEL_NAME = "../../model/Qwen2.5-1.5B-Instruct"
EMB_MODEL_NAME = "../../model/bge-large-en-v1.5"
# define the absolute path of the LLM model
MODEL_PATH = os.path.abspath(MODEL_NAME)
EMB_MODEL_PATH = os.path.abspath(EMB_MODEL_NAME)


# define the absolute path of the LLM model
print(f"Loading HybridLLMEvaluator ... \
        \n- MODEL_NAME={MODEL_PATH}\n- EMB_MODEL_NAME={EMB_MODEL_PATH}")

evaluator = HybridLLMEvaluator(
    embedding_model_name=EMB_MODEL_PATH,
    judge_model_name=MODEL_PATH,
)

### Run the experiment

In [ ]:
import json
import logging
import os
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, ValidationError
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# load modules/prompts
import llm_evaluation

importlib.reload(llm_evaluation)
from llm_evaluation import  HybridLLMEvaluator


# RUN EXPERIMENT
start_time = time.time()

# =========================
# MLflow RUN
# =========================

with mlflow.start_run():

    mlflow.log_param("model", MODEL_NAME)
    mlflow.log_param("system", "clean_two_model_pipeline")

    SOURCE_TEXT = """
    Apple released a new AI-enabled MacBook Pro
    featuring improved battery life and a faster M4 chip.
    """

    EXTRACTION = {
        "summary": (
            "Apple released a new MacBook Pro "
            "with an M4 chip and better battery life."
        ),
        "entities": [
            "Apple",
            "MacBook Pro",
            "M4 chip",
        ],
        "key_points": [
            "AI-enabled laptop",
            "Improved battery",
            "Faster processor",
        ],
    }

    EXTRACTION_RUNS = [
        json.dumps(EXTRACTION),
        json.dumps(EXTRACTION),
    ]

    # evaluator = HybridLLMEvaluator(embedding_model_name = EMB_MODEL_PATH, judge_model_name = MODEL_PATH,)

    result = evaluator.evaluate(
        sample_id="sample_001",
        source_text=SOURCE_TEXT,
        extraction=EXTRACTION,
        extraction_runs=EXTRACTION_RUNS,
    )

    print("\nEvaluation Result:")
    print(json.dumps(asdict(result), indent=2))


total_latency = time.time() - start_time

print("\n===== ELAPSED TIME =====\n")
print(f"{total_latency:.2f} seconds")

# Syntax

In [ ]:
# Extract JSON from markdown code blocks if present
if "```json" in extracted:
    result = "```json" + extracted.split("```json")[1].strip()
elif "```" in extracted:
    result = "```" + extracted.split("```")[1].strip()
result